In [ ]:
import os, sys
import pickle

import matplotlib_inline
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec
from matplotlib import ticker
from matplotlib.lines import Line2D

from tqdm import tqdm

from scipy.interpolate import interp1d
import numpy as np
import pymaster as nmt

matplotlib_inline.backend_inline.set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2
# Load plot settings
import healpy as hp
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
hp.gnomview(np.where(0==old_map*mask, np.nan, old_map), xsize=2000,  rot=(260, -70), norm='hist')

In [ ]:
hp.gnomview(np.where(0==new_map*mask, np.nan, new_map), xsize=2000,  rot=(260, -70), norm='hist')

In [ ]:
plt.hist(np.where(0==old_map*mask, np.nan, old_map), bins=100, label='Old ILC map');
plt.hist(np.where(0==new_map*mask, np.nan, new_map), alpha=0.5, bins=100, label='New ILC map');
plt.xlim(-1e-4, 1e-4);
plt.legend();

In [ ]:
mcols_default[1]

In [ ]:
## load foregrounds
foregrounds_266 = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/foregrounds_0.266.fits")
foregrounds_502 = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/foregrounds_0.502.fits")
foregrounds_798 = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/foregrounds_0.798.fits")
foregrounds_1094 = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/foregrounds_1.094.fits")
foregrounds_1390 = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/foregrounds_1.390.fits")

In [ ]:
import matplotlib.patches as mpatches

for i, freq in enumerate([266, 502, 798, 1094, 1390]):
    fg_map = eval(f"foregrounds_{freq}")
    hp.mollview(fg_map, title=None, cbar=None, norm="hist", cmap="plasma")
   # Make the axes (and healpy-specific background) transparent
    ax = plt.gca()
    ax.patch.set_alpha(0)
    ax.patch.set_facecolor('none')
    if hasattr(ax, "background_patch"):
        ax.background_patch.set_alpha(0)
        ax.background_patch.set_facecolor('none')
    if hasattr(ax, "outline_patch"):
        ax.outline_patch.set_visible(False)
    
plt.savefig("plots/foregrounds_{}MHz.png".format(freq), dpi=500, bbox_inches='tight', transparent=True)

In [ ]:
# load ILC map
mask = hp.read_map("/usr3/graduate/ebaker/dark_photon_constraints/ilc/pyilc_files/roman_mask_20deg.fits")
ilc_map = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src/clean_needlet_ILC_map_roman_20deg_01Kthermal_mK.fits")
hp.mollview(np.where(mask==0, np.nan, 1) * ilc_map,  title=None, cbar=None, norm="hist", cmap="plasma")
plt.savefig("plots/ilc_map.png", bbox_inches='tight', dpi=500, transparent=True)

In [ ]:
# load ILC map
galaxy_map = hp.read_map("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/halo_data/correct_virial_mass/smooth_gal_map_K22.fits")
hp.mollview(galaxy_map,  title=None, cbar=None, norm="hist", cmap="viridis")
plt.savefig("plots/galaxy_map.png", bbox_inches='tight', dpi=500, transparent=True)

In [ ]:
PP_Cls.shape

In [ ]:
# load power spectra
binning = nmt.NmtBin.from_nside_linear(2048, 150)
radio = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_0.1mJy_new_freq/cross_Pg_Cls_binning150_K22.npy")
radio_covs = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_0.1mJy_new_freq/covariance_clean_gP_binning150_K22.npy")
signal = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/halo_data/correct_virial_mass/Cl_gP_mA5.623e-13_modelK22.npy")
signal_ls = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/halo_data/correct_virial_mass/ls.npy")

In [ ]:
new_prefac = 2* np.pi * 410e6 * 6.58212e-16 # convert MHz to eV
plt.errorbar(
    binning.get_effective_ells(),
    radio/new_prefac,
    yerr= np.sqrt(np.diag(radio_covs))/new_prefac,
    capsize=5,
    linestyle=" ",
    marker=".",
    label="ILC Power Spectrum",
    # color=cols_default[4],
)

plt.plot(
    signal_ls,
    -(2.73e3) * (3e-8)**2 * signal/new_prefac,
    # color="black",
    label="Theoretical Signal"
)



theory_line = Line2D([0,1],[0,1],linestyle='-', color='k')
legend2 = plt.legend([theory_line], [r"Theoretical Signal"], 
                     fontsize=22, loc="lower left")

# plt.gca().add_artist(legend2)
legend1 = plt.legend(fontsize=20, framealpha=0.5) # loc='center left', bbox_to_anchor=(1, 0.5))

# plt.xscale("log")
# plt.yscale("log")
plt.xlim(100, 4000)
plt.ylim(-0.2e-12/new_prefac, 0.2e-12/new_prefac)
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C^{\rm Tg}_\ell$ [mK]")
plt.savefig("plots/combined_basic_Cls.svg", bbox_inches="tight")

In [ ]:
new_prefac = 2* np.pi * 266e6 * 6.58212e-16 # convert MHz to eV
plt.errorbar(
    binning.get_effective_ells(),
    radio/new_prefac,
    yerr= np.sqrt(np.diag(radio_covs))/new_prefac,
    capsize=5,
    linestyle=" ",
    marker=".",
    # label="ILC Power Spectrum",
    color=cols_default[0],
)


# theory_line = Line2D([0,1],[0,1],linestyle='-', color='k')
# legend2 = plt.legend([theory_line], [r"Theoretical Signal"], 
                    #  fontsize=22, loc="lower left")

# plt.gca().add_artist(legend2)
# legend1 = plt.legend(fontsize=20, framealpha=0.5) # loc='center left', bbox_to_anchor=(1, 0.5))

# plt.xscale("log")
# plt.yscale("log")
plt.xlim(100, 4000)
plt.ylim(-0.2e-12/new_prefac, 0.2e-12/new_prefac)
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C^{\rm Tg, obs}_\ell$ [mK]")
plt.savefig("plots/observed_basic_Cls.svg", bbox_inches="tight")

In [ ]:
new_prefac = 2* np.pi * 266e6 * 6.58212e-16 # convert MHz to eV


plt.plot(
    signal_ls,
    -(2.73e3) * (3e-8)**2 * signal/new_prefac,
    # color="black",
    label="Theoretical Signal",
    color=cols_default[1],
)



theory_line = Line2D([0,1],[0,1],linestyle='-', color='k')
# legend2 = plt.legend([theory_line], [r"Theoretical Signal"], 
                    #  fontsize=22, loc="lower left")

# plt.gca().add_artist(legend2)
# legend1 = plt.legend(fontsize=20, framealpha=0.5) # loc='center left', bbox_to_anchor=(1, 0.5))

# plt.xscale("log")
# plt.yscale("log")
plt.xlim(100, 4000)
plt.ylim(-0.2e-12/new_prefac, 0.2e-12/new_prefac)
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C^{\rm Tg, pred}_\ell$ [mK]")
plt.savefig("plots/predicted_basic_Cls.svg", bbox_inches="tight")